In [1]:
import pandas as pd
from pymatgen.core import Composition
from matminer.featurizers.composition import ElementProperty
import os

/opt/homebrew/Cellar/jupyterlab/4.5.8_1/libexec/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import os
os.chdir("/Users/anitanam/optical_materials_ml")
print(os.getcwd())

/Users/anitanam/optical_materials_ml


In [5]:
import pandas as pd
from pymatgen.core import Composition
from matminer.featurizers.composition import ElementProperty

jarvis = pd.read_parquet("data/raw/jarvis_dft3d.parquet")
jarvis.shape

(75993, 12)

In [6]:
def to_reduced_formula(formula):
    try:
        return Composition(formula).reduced_formula
    except:
        return None

jarvis["reduced_formula"] = (
    jarvis["formula"]
    .apply(to_reduced_formula)
)

jarvis["reduced_formula"].isna().sum()

/opt/homebrew/Cellar/python@3.14/3.14.6/Frameworks/Python.framework/Versions/3.14/lib/python3.14/functools.py:1126: UserWarning: No Pauling electronegativity for He. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  val = self.func(instance)
/opt/homebrew/Cellar/python@3.14/3.14.6/Frameworks/Python.framework/Versions/3.14/lib/python3.14/functools.py:1126: UserWarning: No Pauling electronegativity for Ar. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  val = self.func(instance)
/opt/homebrew/Cellar/python@3.14/3.14.6/Frameworks/Python.framework/Versions/3.14/lib/python3.14/functools.py:1126: UserWarning: No Pauling electronegativity for Ne. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  val = self.func(instance)


np.int64(0)

In [7]:
jarvis["reduced_formula"].nunique()

51889

In [8]:
bandgap_df = jarvis[
    ["reduced_formula", "optb88vdw_bandgap"]
].copy()

bandgap_df = bandgap_df.dropna(
    subset=["optb88vdw_bandgap"]
)

bandgap_df = (
    bandgap_df
    .groupby(
        "reduced_formula",
        as_index=False
    )
    .median(numeric_only=True)
)

bandgap_df.shape

(51889, 2)

In [9]:
bandgap_df["composition"] = (
    bandgap_df["reduced_formula"]
    .apply(Composition)
)

bandgap_df[
    ["reduced_formula", "composition"]
].head()

,reduced_formula,composition
0,Ac,(Ac)
1,Ac(AgGe)2,"(Ac, Ag, Ge)"
2,Ac(AlAg)2,"(Ac, Al, Ag)"
3,Ac(AlSi)2,"(Ac, Al, Si)"
4,Ac(BIr)2,"(Ac, B, Ir)"


In [10]:
sample = bandgap_df.head(10).copy()

featurizer = ElementProperty.from_preset(
    "magpie"
)

sample_features = featurizer.featurize_dataframe(
    sample,
    col_id="composition",
    ignore_errors=True
)

sample_features.shape

ElementProperty: 100%|████████████████████████████████████████████| 10/10 [00:00<00:00, 123.07it/s]


(10, 135)

In [11]:
bandgap_features = featurizer.featurize_dataframe(
    bandgap_df.copy(),
    col_id="composition",
    ignore_errors=True,
    pbar=True
)

bandgap_features.shape

ElementProperty: 100%|███████████████████████████████████████| 51889/51889 [23:01<00:00, 37.57it/s]


(51889, 135)

In [12]:
import os

os.makedirs(
    "notebooks/data/processed",
    exist_ok=True
)

bandgap_features.to_parquet(
    "notebooks/data/processed/bandgap_features.parquet",
    index=False
)

ArrowInvalid: ("Could not convert Composition('Ac1') with type Composition: did not recognize Python value type when inferring an Arrow data type", 'Conversion failed for column composition with type object')

In [13]:
import os

os.path.exists(
    "notebooks/data/processed/bandgap_features.parquet"
)

True

In [14]:
bandgap_features = bandgap_features.drop(
    columns=["composition"]
)

bandgap_features.shape

(51889, 134)

In [15]:
import os

os.makedirs(
    "notebooks/data/processed",
    exist_ok=True
)

bandgap_features.to_parquet(
    "notebooks/data/processed/bandgap_features.parquet",
    index=False
)

In [16]:
import os

os.path.exists(
    "notebooks/data/processed/bandgap_features.parquet"
)

True

In [17]:
import pandas as pd

check_df = pd.read_parquet(
    "notebooks/data/processed/bandgap_features.parquet"
)

check_df.shape

(51889, 134)